In [1]:
import pandas as pd
import numpy as np
import joblib, json
from sklearn.model_selection import train_test_split

# Reload the original data exactly as in the notebook
data_dl = pd.read_csv('data/data_dl.csv')



In [2]:
# Recreate the same null fix
def null_fix(df):
    for col in df.columns:
        if col == "Monthly_Inhand_Salary" and df[col].isnull().any():
            df[col] = df['Annual_Income'] / 12
        elif df[col].isnull().any():
            df[col].fillna(0, inplace=True)
    return df



In [3]:
data_dl = data_dl.pipe(null_fix)

label_map = {"Poor": 0, "Standard": 1, "Good": 2}
data_dl["Target_label"] = data_dl["Target"].map(label_map)
target_cols = ['Target_label']



/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_81402/3098525870.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)


In [4]:
# Reproduce the exact same split — same random_state=42 gives identical indices
_, X_test_raw, _, y_test_raw = train_test_split(
    data_dl.drop(columns=target_cols),
    data_dl[target_cols],
    test_size=0.25,
    random_state=42,
    stratify=data_dl[target_cols]
)


In [5]:

# X_test_raw.index holds the original row positions from data_dl
# Use those as user IDs — or pull Customer_ID if it exists as a column
if 'Customer_ID' in X_test_raw.columns:
    user_ids = X_test_raw['Customer_ID'].astype(str).tolist()
else:
    # Fallback: synthesise IDs from the original dataframe index
    user_ids = [f"USR_{i}" for i in X_test_raw.index]

print(f"Test set size: {len(X_test_raw)}")
print(f"Sample IDs: {user_ids[:5]}")
print(f"Columns: {X_test_raw.columns.tolist()}")

Test set size: 3125
Sample IDs: ['USR_11847', 'USR_7174', 'USR_125', 'USR_7241', 'USR_6287']
Columns: ['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'Target', 'loan_type__auto_loan', 'loan_type__builder_loan', 'loan_type__credit', 'loan_type__debt_consolidation_loan', 'loan_type__home_equity_loan', 'loan_type__mortgage_loan', 'loan_type__not_specified', 'loan_type__payday_loan', 'loan_type__personal_loan', 'loan_type__student_loan', 'month2_balance', 'month2_credit_util', 'month2_delayed_payment', 'month2_delay_from_due_date', 'month3_balance', 'month3_credit_util', 'month3_delayed_payment', 'month3_delay_fr

In [6]:
# Load the saved preprocessors
scaler  = joblib.load('artefacts/scaler.pkl')
le_dict = joblib.load('artefacts/le_dict.pkl')



In [7]:
# Identify column types (same as notebook)
num_cols = data_dl.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = data_dl.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ('Target', 'Target_label', 'Customer_ID')]

# X_test_raw still has ORIGINAL (pre-encoding, pre-scaling) values
# because we split before encoding — so no decoding needed
sample = X_test_raw.copy()



In [8]:
# Drop any non-feature columns
drop_cols = [c for c in ['Target', 'Target_label', 'Customer_ID'] if c in sample.columns]
feature_cols = [c for c in sample.columns if c not in drop_cols]
sample = sample[feature_cols]

# Write test_user_ids.json
test_users = []
for uid, (_, row) in zip(user_ids, sample.iterrows()):
    features = {}
    for col in feature_cols:
        val = row[col]
        # Convert numpy types to plain Python for JSON serialisation
        if isinstance(val, (np.integer,)):
            val = int(val)
        elif isinstance(val, (np.floating,)):
            val = float(val)
        features[col] = val
    test_users.append({"user_id": uid, "features": features})



In [9]:
with open('test_user_ids.json', 'w') as f:
    json.dump(test_users, f, indent=2)

print(f"Written {len(test_users)} users to test_user_ids.json")
print("Sample entry:")
print(json.dumps(test_users[0], indent=2))

Written 3125 users to test_user_ids.json
Sample entry:
{
  "user_id": "USR_11847",
  "features": {
    "Age": 20,
    "Occupation": "Mechanic",
    "Annual_Income": 20858.03,
    "Monthly_Inhand_Salary": 1738.1691666666666,
    "Num_Bank_Accounts": 10,
    "Num_Credit_Card": 7,
    "Interest_Rate": 32,
    "Num_of_Loan": 2,
    "Delay_from_due_date": 54,
    "Num_of_Delayed_Payment": 25.0,
    "Changed_Credit_Limit": 3.25,
    "Num_Credit_Inquiries": 13.0,
    "Credit_Mix": "Bad",
    "Outstanding_Debt": 2164.84,
    "Credit_Utilization_Ratio": 22.869052141546387,
    "Credit_History_Age": 147,
    "Payment_of_Min_Amount": "Yes",
    "Total_EMI_per_month": 30.79305493700541,
    "Amount_invested_monthly": 70.9851731300151,
    "Payment_Behaviour": "Low_spent_Small_value_payments",
    "Monthly_Balance": 341.0386885996461,
    "loan_type__auto_loan": 0.0,
    "loan_type__builder_loan": 0.0,
    "loan_type__credit": 0.0,
    "loan_type__debt_consolidation_loan": 0.7087798660532137,
    "